# 23 - Faa di Bruno multi-layer jets

omnibias computes the **exact** activation derivative tower `sigma^(k)` in
closed form. For a *single* layer the pre-activation is affine, so higher
derivatives are trivial. The interesting case is **composition of many
nonlinear layers** - that is exactly Faa di Bruno's formula.

This notebook propagates a truncated Taylor **jet** along a line
`x(t) = x0 + t v` through a deep MLP with `omnibias.jax.mlp_jet`, giving the
exact directional derivatives `d^k/dt^k f(x0 + t v)` with **no nested
autodiff and no finite differences**. We then:

1. verify machine-precision agreement with nested autodiff,
2. show finite differences fall apart at high order while the exact jet does not,
3. recover a full **Hessian** of a deep net by polarization of directional 2-jets.

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from _style import set_style, PRIMARY, ACCENT, GOOD, WARM
set_style()

from omnibias.jax import mlp_jet, jet_to_tower
from omnibias.jax.activations import get_activation
from omnibias.core.bell import faa_di_bruno_terms

rng = np.random.default_rng(0)
print('jax', jax.__version__)

## A deep MLP and its exact directional jet

We build a 4-layer `tanh` MLP `R^4 -> R^3` and a random base point/direction.
`mlp_jet` returns Taylor coefficients; `jet_to_tower` rescales them to the
derivative tower `(f, f', f'', ...)` along the line.

In [ ]:
dims = (4, 16, 12, 3)
tanh = get_activation('tanh')
layers = []
for i in range(len(dims) - 1):
    W = jnp.asarray(rng.normal(scale=0.8, size=(dims[i+1], dims[i])))
    b = jnp.asarray(rng.normal(scale=0.3, size=(dims[i+1],)))
    spec = None if i == len(dims) - 2 else tanh
    layers.append((W, b, spec))

x0 = jnp.asarray(rng.normal(size=(dims[0],)))
v = jnp.asarray(rng.normal(size=(dims[0],)))
v = v / jnp.linalg.norm(v)

def f(x):
    z = x
    for W, b, spec in layers:
        z = W @ z + b
        if spec is not None:
            z = spec.forward(z)
    return z

order = 8
tower = jet_to_tower(mlp_jet(x0, v, layers, order))   # (order+1, 3)
for k in range(5):
    print(f'd^{k}/dt^{k} f(x0)  =', np.round(np.asarray(tower[k]), 5))

## Exact jet vs nested autodiff vs finite differences

Nested autodiff (repeated `jax.jacfwd`) is the ground truth. Central finite
differences of order `k` divide by `h^k`, so catastrophic cancellation in
float64 makes them useless beyond a few orders. The closed-form jet stays at
machine precision for every order.

In [ ]:
from math import comb

def g(t):
    return f(x0 + t * v)

def ad_kth(k):
    fn = g
    for _ in range(k):
        fn = jax.jacfwd(fn)
    return np.asarray(fn(0.0))

def fd_kth(k, h=1e-2):
    # central finite difference of the k-th derivative (component-wise)
    acc = 0.0
    for j in range(k + 1):
        acc = acc + (-1)**j * comb(k, j) * np.asarray(g((k/2 - j) * h))
    return acc / h**k

ad = np.stack([ad_kth(k) for k in range(order + 1)])
jet_err = np.abs(np.asarray(tower) - ad).max(axis=1)
fd_err = np.array([np.abs(fd_kth(k) - ad[k]).max() for k in range(order + 1)])

ks = np.arange(order + 1)
fig, ax = plt.subplots()
ax.semilogy(ks, np.clip(fd_err, 1e-20, None), 'o-', color=ACCENT, label='finite difference')
ax.semilogy(ks, np.clip(jet_err, 1e-20, None), 's-', color=GOOD, label='omnibias exact jet')
ax.set_xlabel('derivative order k'); ax.set_ylabel('max abs error vs autodiff')
ax.set_title('Exact jets stay at machine precision; FD blows up')
ax.legend(); fig.tight_layout()
print('max jet error over all orders:', jet_err.max())

## Why high order is cheap: the Faa di Bruno term count

The naive Bell-polynomial sum has one term per integer partition of `n`,
which grows fast. The shifted-power kernel omnibias actually uses is only
`O(n^2)` per element - it never enumerates these partitions (they are used
only as an independent test oracle).

In [ ]:
counts = [len(faa_di_bruno_terms(n)) for n in range(1, order + 1)]
fig, ax = plt.subplots()
ax.bar(range(1, order + 1), counts, color=PRIMARY)
ax.set_xlabel('order n'); ax.set_ylabel('# Faa di Bruno terms (partitions of n)')
ax.set_title('Partition explosion avoided by the shifted-power kernel')
fig.tight_layout()
print('term counts:', counts)

## A deep closed-form Hessian by polarization

A directional 2-jet gives `v^T H v` exactly. The full Hessian of a scalar
output follows from the polarization identity

$$ 2\, e_i^T H e_j = (e_i+e_j)^T H (e_i+e_j) - e_i^T H e_i - e_j^T H e_j, $$

each term a single order-2 directional jet. We reconstruct the Hessian of the
first output component and check it against `jax.hessian`.

In [ ]:
D = dims[0]
comp = 0

def dir2(u):
    u = jnp.asarray(u)
    return jet_to_tower(mlp_jet(x0, u, layers, 2))[2, comp]   # u^T H u

eye = np.eye(D)
diag = np.array([dir2(eye[i]) for i in range(D)])
H = np.zeros((D, D))
for i in range(D):
    H[i, i] = diag[i]
    for j in range(i + 1, D):
        quad = dir2(eye[i] + eye[j])
        H[i, j] = H[j, i] = 0.5 * (quad - diag[i] - diag[j])

H_ad = np.asarray(jax.hessian(lambda x: f(x)[comp])(x0))
print('max abs Hessian error vs jax.hessian:', np.abs(H - H_ad).max())

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6))
for ax, M, ttl in zip(axes, [H, H_ad], ['polarized jet Hessian', 'jax.hessian']):
    im = ax.imshow(M, cmap='RdBu_r'); ax.set_title(ttl); fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()

## Takeaway

- `mlp_jet` propagates an exact Taylor jet through arbitrarily deep
  compositions using omnibias' closed-form `sigma^(k)` and Faa di Bruno.
- It matches nested autodiff to ~1e-12 at order 8, where finite differences
  have already diverged by many orders of magnitude.
- Directional jets + polarization give exact higher-order objects (Hessians,
  and beyond) of deep networks without nested autodiff.
- The same `omnibias.torch.mlp_jet` is a bit-identical twin.